<a href="https://colab.research.google.com/github/kjfcvx12/Colab/blob/main/05_04_01_clip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# 필요한 패키지 설치 (ftfy, regex, tqdm)
os.system("pip install ftfy regex tqdm")

# GitHub에서 openai의 CLIP 리포지토리 직접 설치
os.system("pip install git+https://github.com/openai/CLIP.git")

In [ ]:
import clip
clip.available_models()

In [ ]:
# clip 모델 객체, 이미지 전처리 함수
model, preprocess = clip.load('ViT-B/32')

In [ ]:
preprocess

In [ ]:
model.cuda().eval()

In [ ]:
model.visual

In [ ]:
# model.visual : clip모델 안에 이미지 처리 전용 신경망 안에
# input_resolution : 입력 이미지 크기
input_resolution=model.visual.input_resolution
print(input_resolution)
print(type(model.visual))

# 모델이 한번에 읽을 수 있는 토큰 (clip은 보통 77토큰)
context_length=model.context_length
print(context_length)

# 모델이 배운 단어의 개수
vocab_size=model.vocab_size
print(vocab_size)

In [ ]:
# 허깅페이스 스타일 모델들 : model.config.vision_config

In [ ]:
text_tokens=clip.tokenize('hi deeplearning')
print(text_tokens)
# 1개 문장(1,77)
print(text_tokens.shape)

In [ ]:
import matplotlib.pyplot as plt
# scikit-image 불러오기
from skimage import data
from PIL import Image
# 파이토치 이미지 전처리
from torchvision import transforms
import numpy as np

# 텍스트 설명
# 각 이미지 이름 : 해당 이미지에 대한 설명
descriptions = {
    "page": "a page of text about segmentation",
    "chelsea": "a facial photo of a tabby cat",
    "astronaut": "a portrait of an astronaut with the American flag",
    "rocket": "a rocket standing on a launchpad",
    "camera": "a person looking at a camera on a tripod",
    "horse": "a black-and-white silhouette of a horse",
    "coffee": "a cup of coffee on a saucer",
     "moon": "a grayscale image of the moon surface"
}

# PyTorch 전처리
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 원본이미지
original_images = []
# 전처리된 Tensor
images = []
# 텍스트 설명 저장
texts = []

image_names = list(descriptions.keys())

plt.figure(figsize=(16, 5))

for i, name in enumerate(image_names):
    img = getattr(data, name)()
    # img=data.horse

    # 흑백 이미지는 3채널로 변환
    # ndim=2 : 흑백이미지(height*width) (100*100)
    # clip/ 컬러이미지 -> 3채널 입력으로 해야함
    if img.ndim == 2:
      # 넘파이로 변경 - 컬러이미지로 [img, img, img] (100, 100, 3)
        img = np.stack([img]*3, axis=-1)

    # bool 타입 이미지는 uint8로 변환
    if img.dtype == np.bool_:
        # 0~255 범위의 이미지 배열
        img = (img * 255).astype(np.uint8)

    # numpy배열 -> PIL이미지
    img_pil = Image.fromarray(img)
    # PIL이미지 -> Tensor
    img_tensor = preprocess(img_pil)

    # 시각화
    plt.subplot(2, 4, i+1)
    plt.imshow(np.array(img_pil))
    plt.title(f"{name}\n{descriptions[name]}")
    plt.xticks([])
    plt.yticks([])

    # 리스트에 저장
    original_images.append(img_pil)
    images.append(img_tensor)
    texts.append(descriptions[name])

plt.tight_layout()
plt.show()


In [ ]:
import torch

# 이미지 리스트를 numpy 배열로 변환한 후, 이를 파이토치 텐서로 변환후 GPU로 전송
image_input = torch.tensor(np.stack(images)).cuda()

# CLIP의 tokenize 함수를 사용하여 텍스트 토큰을 생성
text_tokens = clip.tokenize(["This is " + desc for desc in texts]).cuda()
text_tokens[0]

In [ ]:
# BPE 알고리즘
# playing -> [1234, 5678]

vocab={
    "a":320,
    "page":1143,
    "of":663,
    "text":34,
    "about":346,
    "segmentation":66,
    "<start_of_token>":49406,
    "<end_of_token>":49407
}

tokens=["<start_of_token>","a","page","of","text","about","segmentation","<end_of_token>"]
token_ids=[vocab[token] for token in tokens]
token_ids

In [ ]:
image_input.shape
#[batch_size, channel, H, W]

In [ ]:
# 이미지를 모델에 넣은 후 shape
# [batch_size, clip vit-b/32 모델의 임베딩 차원수]
image_features=model.encode_image(image_input).float()
image_features.shape

In [ ]:
# 텍스트를 모델에 넣은 후 shape
# 두 벡터 공간이 같으나 차원 -> consine similarity 계산 가능해짐
text_features=model.encode_text(text_tokens).float()
text_features.shape

In [ ]:
with torch.no_grad():
  # 이미지를 모델에 넣은 -> 이미지 특성 추출됨
  image_features=model.encode_image(image_input).float()
  text_features=model.encode_text(text_tokens).float()

In [ ]:
# 각 이미지 벡터를 자기 자신의 길이로 나눔 -> 길이 1로 정규화
# 벡터의 방향만 남기기 위한 작업
# 유사도 측정 시 벡터의 크기 너무 크면 결과가 왜곡되니까
image_features /= image_features.norm(dim=-1, keepdim=True)
text_features /= text_features.norm(dim=-1, keepdim=True)

In [ ]:
# cpu().numpy() : 텐서 cpu로 옮기고 numpy 배열로 반환
similarity=text_features.cpu().numpy() @ image_features.cpu().numpy().T

In [ ]:
image_features.shape, text_features.shape, similarity.shape

# [8, 512] @ [8, 512]

In [ ]:
count = len(descriptions)

plt.figure(figsize=(20, 14))

# 이미지와 텍스트 사이의 코사인 유사도를 표시
plt.imshow(similarity, vmin=0.1, vmax=0.3)
plt.yticks(range(count), texts, fontsize=18)
plt.xticks([])

# 각 이미지가 위로 표시
for i, image in enumerate(original_images):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")

# 유사도 매트릭스의 각 셀에 해당하는 값을 표시
# 열의 개수(이미지 총 개수)
for x in range(similarity.shape[1]):
  # 텍스트 문자의 총 개수
    for y in range(similarity.shape[0]):
      # (x,y) 좌표에 해당 유사도 점수를 텍스트로 넣음
        plt.text(x, y, f"{similarity[y, x]:.2f}", ha="center", va="center", size=12)

# 그래프 주변의 테두리 제거
for side in ["left", "top", "right", "bottom"]:
  plt.gca().spines[side].set_visible(False)

# x축과 y축의 범위 설정
plt.xlim([-0.5, count - 0.5])
plt.ylim([count + 0.5, -2])

# 그래프 제목 설정
plt.title("Cosine similarity between text and image features", size=20)

In [ ]:
# hugging fase의 datasets 라이브러리 사용함
!pip install datasets

In [ ]:
from datasets import load_dataset

In [ ]:
cifar100=load_dataset("cifar100")

In [ ]:
cifar100

In [ ]:
print(cifar100['train'].features['fine_label'])